# 05.01 模块与包：把自己的代码变成"可复用的工具"

### 学习目标

1. 分清 **模块(module) / 包(package) / 库(library)** 这三个词
2. 掌握 `import` 的三种写法，并知道什么时候用哪种
3. 理解 `sys.path` 是什么——以及**"为什么 import 会失败"**
4. 认识 `__init__.py` 的作用
5. 亲手导入本项目自带的工具包 `my_uyils`，把"自己写的代码"变成"能 import 的东西"

### 前置知识

需要你先学过：函数、`if __name__ == '__main__'`、相对路径。
如果这些还不熟，先回去看 `03` 和 `04` 两个模块。


## 一、先把三个容易混的词分清

| 名词 | 是什么 | 本项目的例子 |
|---|---|---|
| **模块** module | 一个 `.py` 文件 | `my_uyils/str_util.py` |
| **包** package | 一个**装了多个模块的文件夹**（带 `__init__.py`） | `my_uyils/` |
| **库** library | 一堆包和模块的**集合**，可以整体安装 | `pandas`、`numpy` |

一句话记忆：**模块是文件，包是文件夹，库是一整套工具箱。**

"模块化"的好处：写一次，到处用。你不需要每次都重新写"反转字符串"这种函数，
只要 `import` 一下，它就出现在你的代码里。


## 二、本笔记要用到的目录结构

我们手里有一个自己写的工具包，长这样（注意它是**本项目的一部分**，不是第三方库）：

```text
05.模块与包/
├── 05.01-模块与包.ipynb     <- 你正在看的这份笔记
├── 生成文件/                  <- 下面演示写文件时，产物会放这里
└── my_uyils/                 <- 一个"包"（文件夹 + __init__.py）
    ├── __init__.py           <- 包的"门面"，负责对外暴露函数
    ├── str_util.py           <- 模块：字符串工具
    └── file_util.py          <- 模块：文件工具
```

### 【关键知识点】为什么带编号的目录不能被 `import`

这是本项目改结构时最容易踩的坑，值得单独讲清楚。

Python 的 `import` 语句要求**路径的每一段都必须是合法的标识符**
（就是"能当变量名"的那种字符串：字母或下划线开头，只能含字母、数字、下划线）。

而我们为了排序清晰，把目录命名成了 `01.基础概念`、`05.模块与包`——
它们**以数字开头**、还**含有点号**，所以：

```python
import 05.模块与包.my_uyils      # SyntaxError！根本写不出来
```

结论：**带 `01.` 这种前缀的目录，只能当"文件夹"用，不能当"包路径"用。**

那怎么办？很简单——**把它加进 `sys.path`，然后直接 import 里面那一层**。
就像下面这样。


### `sys.path` 是什么？

你每次写 `import xxx`，Python 都会去一张"清单"里挨个目录找 `xxx`。
这张清单就是 `sys.path`。它是一个普通的列表，**你可以随时往里加目录**。

只要把 `my_uyils` 的**上一层目录**加进 `sys.path`，
Python 就能找到 `my_uyils` 这个包了。


In [ ]:
import os
import sys
from pathlib import Path


def find_pkg_root(name='my_uyils'):
    """在几个候选目录里找 name/，返回它所在的目录。

    为什么要写得这么啰嗦？因为 Jupyter 的工作目录可能是
    notebook 所在目录，也可能是启动 jupyter 的目录，两种情况都要兜住。
    """
    here = Path.cwd()
    candidates = [
        here,                          # 工作目录 == 本笔记目录
        here / '05.模块与包',            # 工作目录 == pythonlearnproject
        here.parent / '05.模块与包',
    ]
    for base in candidates:
        if (base / name).is_dir():
            return base
    raise FileNotFoundError(f'找不到 {name}/，当前工作目录是 {here}')


PKG_ROOT = find_pkg_root()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))       # 【关键一步】加进搜索路径

print('工具包所在目录：', PKG_ROOT)
print('sys.path 的前两项：')
for p in sys.path[:2]:
    print('   ', p)


现在 `my_uyils` 就在搜索路径里了，下面三种导入写法都能用了。

---

## 三、`import` 的三种写法

从"啰嗦但清楚"到"简洁但需要包作者配合"，三种各有用途。

### 写法 1：导入模块，用时带前缀

```python
import my_uyils.str_util              # 只导入模块
my_uyils.str_util.str_reverse('abc')  # 用的时候写全
```

**好处**：一眼看出这个函数是从哪来的，不容易和别的模块重名。
**坏处**：名字太长，写起来累。


In [ ]:
import my_uyils.str_util as str_util
import my_uyils.file_util as file_util

print('=== 写法 1：带前缀调用 ===')
print(str_util.str_reverse('新年快乐'))
print(str_util.substr('hape', 0, 2))


### 写法 2：从模块里导入具体函数（日常最常用）

```python
from my_uyils.str_util import str_reverse
str_reverse('abc')        # 直接用，不用前缀
```

**好处**：简洁。
**坏处**：名字多了容易忘记出处，也容易和本文件的函数重名。


In [ ]:
from my_uyils.str_util import substr, count_chars

print('\n=== 写法 2：导入具体函数 ===')
print('截取：', substr('hape', 0, 2))
print('字符统计：', count_chars('aabbb'))


### 写法 3：利用 `__init__.py` 预导入（最简洁）

打开 `my_uyils/__init__.py` 你会看到：

```python
from .str_util import str_reverse, substr, is_palindrome, count_chars
from .file_util import print_file_info, append_to_file
```

这叫做**预导入**：包的作者把常用函数提前"摆到门口"，
使用者就可以少写一层路径，直接从包名导入。

其中开头的 `.` 表示"**当前包**"，这叫**相对导入**——它保证了包被移来移去也不会出错。


In [ ]:
from my_uyils import str_reverse, is_palindrome, count_chars

print('\n=== 写法 3：从包直接导入（预导入的功劳）===')
print('反转：', str_reverse('abcdefg'))
print('回文判断：', is_palindrome('上海自来水来自海上'))
print('字符统计：', count_chars('aabbb'))


### `__init__.py` 到底是什么？

- 它的存在，是把一个普通文件夹**标记成"包"**的标志（Python 3.3+ 其实允许没有，
  但**有它更明确**，也能写初始化代码，推荐保留）。
- 它在包**第一次被 import 时自动执行一次**。
- 最常见的用法就是上面这种"预导入"，或者放 `__all__` 控制 `from 包 import *` 的行为。


## 四、文件工具：顺便讲清"文件到底写到哪去了"

下面用 `file_util` 里的两个函数演示写文件。

这里藏着一个初学者几乎都会踩的坑：**相对路径是相对于"当前工作目录"，不是相对于"脚本文件"**。

| 你写的路径 | 实际含义 | 危险在哪 |
|---|---|---|
| `'data.txt'` | 当前**工作目录**下的 data.txt | 换个目录启动，文件就跑到别处去了 |
| `os.path.join(BASE, 'data.txt')` | 由 BASE 决定的绝对位置 | 明确、可控 |

在**脚本**里你可以用 `__file__` 拿到脚本自己的位置。
但在 **notebook 里没有 `__file__` 这个变量**，所以 notebook 里通常直接用工作目录，
或者像下面这样显式指定一个输出文件夹。

本项目统一约定：**所有演示产物都写进 `生成文件/`，不往项目根目录乱丢。**


In [ ]:
OUT_DIR = os.path.join(os.getcwd(), '生成文件')
os.makedirs(OUT_DIR, exist_ok=True)          # 不存在就创建，存在也不报错

target = os.path.join(OUT_DIR, 'demo_output.txt')

file_util.append_to_file(target, 'Hello World')
file_util.append_to_file(target, 'happy everyday')
print(f'已写入：{target}')

file_util.print_file_info(target)


## 五、小结：什么时候用哪种写法？

| 写法 | 写法示例 | 适用场景 |
|---|---|---|
| 1 完整前缀 | `import my_uyils.str_util` | 模块名不长，或想保留"来自哪个模块"的信息 |
| 2 导入函数 | `from my_uyils.str_util import f` | **日常最常用**，缺什么导什么 |
| 3 预导入 | `from my_uyils import f` | 包作者贴心地摆好了门面，少打字 |

### 常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 目录名以数字开头 / 含点号 | `SyntaxError`，根本写不出 import 语句 | 把上层目录加进 `sys.path`，只 import 合法的那一层 |
| 忘记把工具包目录加进 `sys.path` | `ModuleNotFoundError: No module named 'my_uyils'` | 先 `sys.path.insert(0, ...)` 再 import |
| 包文件夹里没有 `__init__.py` | 预导入失效，`from 包 import 函数` 报错 | 补上 `__init__.py` 并写好预导入 |
| 写文件用了裸相对路径 | 文件"不知道跑哪去了" | 用 `os.path.join(明确的目录, 文件名)` |
| notebook 里用 `__file__` | `NameError: name '__file__' is not defined` | notebook 里改用 `os.getcwd()` |
| 包内部用了 `import str_util`（绝对导入） | 包被移动后报错 | 包内部一律用相对导入 `from .str_util import ...` |


## 六、练习

1. **加一个函数**：给 `str_util.py` 加一个 `title_case(s)`，把每个单词的首字母大写
   （提示：可以用 `s.title()`，也可以自己写）
2. **找出出现最多的字符**：用 `count_chars()` 的结果，找出次数最多的那个字符
3. **改个名字**：把 `my_uyils` 文件夹改名为 `my_utils`（修掉这个笔误），
   然后让本笔记依然能正常导入——想想你需要改哪几个地方？
4. **新增一个模块**：在包里加一个 `math_util.py`，写一个 `is_prime(n)` 判断质数，
   并把它也加进 `__init__.py` 的预导入里


### 练习答案思路（先自己做，再看）

**练习 2** 的关键是：`count_chars` 返回的是一个字典，用 `max` 配合 `key=` 取值。

**练习 3** 需要改的地方（想全了才算过关）：
- 文件夹本身的改名
- `__init__.py` 里的相对导入**不用改**（相对导入正是为了应对这种情况）
- 本笔记里 `find_pkg_root('my_uyils')` 的默认参数
- 笔记里的 `import my_uyils...` 全部要跟着改

这正是**相对导入**的价值：包内部互相引用不写死包名，改名时改动最少。


In [ ]:
# 练习 2 的参考写法（取消注释即可运行）：
# counts = count_chars('aabbbccdd')
# top = max(counts, key=counts.get)
# print(f'出现最多的是 {top!r}，共 {counts[top]} 次')

# 检查一下刚才的产物还在不在
print('\n生成文件目录内容：')
for p in sorted(Path(OUT_DIR).iterdir()):
    print('   ', p.name)
